# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here
df = spark.sql("""
    SELECT facid, SUM(slots) AS `Total Slots`
    FROM bookings
    WHERE starttime >= '2012-09-01' AND starttime < '2012-10-01'
    GROUP BY facid
    ORDER BY `Total Slots`
""")
df.show()
df.write.parquet("month.parquet")


+-----+-----------+
|facid|Total Slots|
+-----+-----------+
|    5|        122|
|    3|        422|
|    7|        426|
|    8|        471|
|    6|        540|
|    2|        570|
|    1|        588|
|    0|        591|
|    4|        648|
+-----+-----------+



## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here
pdf = spark.sql("""
                select distinct mems.firstname || ' ' || mems.surname as member, facs.name as facility
	from 
		members mems
		inner join bookings bks
			on mems.memid = bks.memid
		inner join facilities facs
			on bks.facid = facs.facid
	where
		facs.name in ('Tennis Court 2','Tennis Court 1')
order by member, facility    
                """)

pdf.show()

pdf.write.partitionBy("facility").format("delta").mode("overwrite").saveAsTable("threejoin_delta")


+--------------+--------------+
|        member|      facility|
+--------------+--------------+
|    Anne Baker|Tennis Court 1|
|    Anne Baker|Tennis Court 2|
|  Burton Tracy|Tennis Court 1|
|  Burton Tracy|Tennis Court 2|
|  Charles Owen|Tennis Court 1|
|  Charles Owen|Tennis Court 2|
|  Darren Smith|Tennis Court 2|
| David Farrell|Tennis Court 1|
| David Farrell|Tennis Court 2|
|   David Jones|Tennis Court 1|
|   David Jones|Tennis Court 2|
|  David Pinker|Tennis Court 1|
| Douglas Jones|Tennis Court 1|
| Erica Crumpet|Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows



## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here
import requests
import pandas as pd
from pyspark.sql.functions import col, weekofyear, max as spark_max

def get_stock_data(symbol):
    url = "https://alpha-vantage.p.rapidapi.com/query"
    
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }
    
    headers = {
        "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
        "X-RapidAPI-Key": "f344ede3famshcfbe67e26491b84p1d77cejsnd90a2a421393"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    # Transform JSON to DataFrame
    df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index')
    df['date'] = df.index
    df['symbol'] = symbol
    return df

# Symbols for the companies
symbols = ["GOOGL", "AAPL", "MSFT", "TSLA"]

# Collect all data
all_data = pd.concat([get_stock_data(symbol) for symbol in symbols])

# Convert to Spark DataFrame
stock_df = spark.createDataFrame(all_data)

# Add week column
stock_df = stock_df.withColumn("week", weekofyear(col("date")))

# Calculate max closing price per week
max_weekly_closing_df = stock_df.groupBy("symbol", "week") \
    .agg(spark_max(col("4. close")).alias("max_closing_price"))

max_weekly_closing_df.write \
    .partitionBy("symbol") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("max_closing_price_weekly")

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-3457612507437688>:33
     30 symbols = ["GOOGL", "AAPL", "MSFT", "TSLA"]
     32 # Collect all data
---> 33 all_data = pd.concat([get_stock_data(symbol) for symbol in symbols])
     35 # Convert to Spark DataFrame
     36 stock_df = spark.createDataFrame(all_data)

File <command-3457612507437688>:33, in <listcomp>(.0)
     30 symbols = ["GOOGL", "AAPL", "MSFT", "TSLA"]
     32 # Collect all data
---> 33 all_data = pd.concat([get_stock_data(symbol) for symbol in symbols])
     35 # Convert to Spark DataFrame
     36 stock_df = spark.createDataFrame(all_data)

File <command-3457612507437688>:24, in get_stock_data(symbol)
     22 data = response.json()
     23 # Transform JSON to DataFrame
---> 24 df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index')
     25 df['date'] = df.index
     26 df['symbol'] = symb

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here

# JDBC connection properties
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

# SQL query to extract 100 RNA records
query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

# Read data from PostgreSQL
rna_df = spark.read.jdbc(url=jdbc_url, table=query, properties=connection_properties)

rna_df.show()

# Save the DataFrame to a managed table
rna_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("rna_100_records")

+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
|      id|          upi|           timestamp|userstamp|           crc64| len|           seq_short|seq_long|                 md5|
+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
| 5110258|URS00004DF9F2| 2014-05-29 13:51:05|   RNACEN|61AE5251E4F4E67A| 848|GATAAACGCTAGCGGAG...|    null|26fa8b5b4fbc5e3bc...|
| 5110259|URS00004DF9F3| 2014-05-29 13:51:05|   RNACEN|63056E024E222787|  76|GCGGGCGTAGCTCAGTT...|    null|f725b5fff34986f21...|
| 5110262|URS00004DF9F6| 2014-05-29 13:51:05|   RNACEN|06115F73D962D669| 693|TGCAGTCGGACGGGATT...|    null|26fa93769ae137737...|
| 5110264|URS00004DF9F8| 2014-05-29 13:51:05|   RNACEN|168BF325E98D59CD|1514|AGAGTTTGATCATGGCT...|    null|be74650d6063133a0...|
| 5110266|URS00004DF9FA| 2014-05-29 13:51:05|   RNACEN|F7A8B3D48878A1EB|1415|GCGGCATGGATTAGGCA...